# Conjunto de teste de reescrita (tarefa 3.1) e normalização da consulta

Cada consulta é uma alegação do corpus **reescrita** como o boato chega ao usuário. O alvo é a alegação original, que continua no índice. Esta parte cobre:

| Categoria | Reescrita | Semente |
|---|---|---|
| `apelido` | 1 entidade trocada por apelido (`dicionarios/apelidos.csv`) | 45 |
| `girias` | internetês (`dicionarios/internetes.csv`), minúsculas, metade das palavras sem acento; mínimo 2 abreviações | 46 |
| `apelido+girias` | as duas juntas, como mensagem de WhatsApp | 47 |
| `negacao` | título de desmentido sem o "não" = o boato como circula ("Lula não disse X; vídeo é editado" → "Lula disse X"); regras em `reescrita.tirar_negacao` | 48 |

E testa a **normalização da consulta** antes da busca: Enelvo (internetês) e o dicionário de apelidos (`normalizacao.py`).

**Índice preliminar:** só as 1.852 alegações de 2022. A avaliação definitiva roda sobre os **dois corpora** (2022 + Central de Fatos), depois da rotulagem entre ciclos.

Saídas: `results/teste_reescrita.csv` (o conjunto, revisável no git), `results/revisao_negacao.csv` (amostra para conferir o gerador de negação) e `results/normalizacao_consulta.csv` (métricas).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from avaliacao import linhas_com_texto, metricas, posicoes
from normalizacao import criar_enelvo, normalizar, normalizar_se_preciso, vocabulario_protegido
from reescrita import (carregar_apelidos, carregar_internetes, erro_digitacao, internetes, tirar_negacao,
                       trocar_apelido)

RESULTS = Path("results")
N_POR_CATEGORIA = 100
MODELO, REVISAO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"

d = pd.read_csv("data/com_texto_limpo.csv")
E_todos = np.load("data/emb_minilm.npy")
assert len(d) == len(E_todos)
d["idx"] = np.arange(len(d))                               # linha original: é o idx_i/idx_j dos pares rotulados
manter = (~d["multi_claim"]).to_numpy()
d, E = d[manter].reset_index(drop=True), E_todos[manter]   # o índice: as mesmas 1.852 alegações do 02
apelidos, tabela_internetes = carregar_apelidos(), carregar_internetes()
print(len(d), "alegações no índice")

/Users/aluno2/Residencia/RES-IA-Challenge-1/.venv-1/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1852 alegações no índice


## 1. Gerar as consultas

Para cada categoria: aplica a reescrita em todas as alegações elegíveis e sorteia 100. Cada alegação-alvo aparece no máximo uma vez por categoria.

In [2]:
def gerar(categoria, semente):
    rng = np.random.default_rng(semente)
    candidatos = []
    for i, alegacao in d["claim"].items():
        if categoria == "negacao":
            texto = tirar_negacao(d.at[i, "Título da checagem"], alegacao)
            if texto is None:
                continue
            candidatos.append((i, texto, ""))
            continue
        if categoria == "digitacao":
            texto, n = erro_digitacao(alegacao, rng)
            if n >= 2:
                candidatos.append((i, texto, f"{n} erros"))
            continue
        if categoria in ("apelido", "apelido+girias"):
            r = trocar_apelido(alegacao, apelidos, rng)
            if r is None:
                continue
            texto, detalhe = r[0], f"{r[1]}→{r[2]}"
        else:
            texto, detalhe = alegacao, ""
        if categoria in ("girias", "apelido+girias"):
            texto, n = internetes(texto, tabela_internetes, rng)
            if n < 2:
                continue
            detalhe = (detalhe + f" | {n} abreviações").strip(" |")
        candidatos.append((i, texto, detalhe))
    escolhidos = rng.choice(len(candidatos), size=min(N_POR_CATEGORIA, len(candidatos)), replace=False)
    print(f"{categoria}: {len(candidatos)} elegíveis, {len(escolhidos)} sorteadas")
    return [{"categoria": categoria, "consulta": candidatos[k][1], "alvo": d.at[candidatos[k][0], "claim"],
             "detalhe": candidatos[k][2], "excluir_texto": ""} for k in escolhidos]


def parafrase_real():
    """Pares reais rotulados 'mesma' no gate: a consulta é uma alegação do próprio índice, então ela
    (e as cópias dela) sai do ranking. Pares de texto idêntico ficam de fora: não há o que medir."""
    p = pd.read_csv("data/pares_temporais.csv")
    v = pd.read_csv(RESULTS / "validacao_pares.csv")
    v["tipo"] = v["tipo"].fillna("").str.strip()
    p = p.merge(v[["idx_i", "idx_j", "tipo"]], on=["idx_i", "idx_j"], how="left")
    texto = d.set_index("idx")["claim"]
    p["consulta"], p["alvo"] = p["idx_i"].map(texto), p["idx_j"].map(texto)
    p = p.dropna(subset=["consulta", "alvo"])
    p = p[p["consulta"].str.lower() != p["alvo"].str.lower()]
    print(f"parafrase_real: {len(p)} pares")
    return [{"categoria": "parafrase_real", "consulta": r.consulta, "alvo": r.alvo,
             "detalhe": r.tipo or "", "excluir_texto": r.consulta} for r in p.itertuples()]


teste = pd.DataFrame(gerar("apelido", 45) + gerar("girias", 46) + gerar("apelido+girias", 47)
                     + gerar("negacao", 48) + gerar("digitacao", 49) + parafrase_real())
apelido_usado = teste["detalhe"].str.extract(r"→([^|]+)")[0].str.strip()
teste["apelido_no_normalizador"] = apelido_usado.map(dict(zip(apelidos["apelido"], apelidos["no_normalizador"])))
teste.insert(0, "consulta_id", range(1, len(teste) + 1))
teste.to_csv(RESULTS / "teste_reescrita.csv", index=False)
teste.groupby("categoria").head(2)[["categoria", "consulta", "alvo"]]

apelido: 930 elegíveis, 100 sorteadas


girias: 500 elegíveis, 100 sorteadas


apelido+girias: 290 elegíveis, 100 sorteadas
negacao: 377 elegíveis, 100 sorteadas
digitacao: 1203 elegíveis, 100 sorteadas
parafrase_real: 34 pares


,categoria,consulta,alvo
0,apelido,Marcelo Madureira não é autor de texto com crí...,Marcelo Madureira não é autor de texto com crí...
1,apelido,Luladrão morreu aos 77 anos de idade após sofr...,Lula morreu aos 77 anos de idade após sofrer AVC
100,girias,"lula naum destratou janja antes do debate, ao ...","Lula não destratou Janja antes do debate, ao c..."
101,girias,a onu naum promove a fome mundial; o artigo q ...,A ONU não promove a fome mundial; o artigo que...
200,apelido+girias,lularapio disse no twitter q vai bloquear poup...,Lula disse no Twitter que vai bloquear poupanç...
201,apelido+girias,pesquisa ipec de 08/2022 naum apontou biroliro...,Pesquisa Ipec de 08/2022 não apontou Bolsonaro...
300,negacao,TSE vetou campanha de vacinação contra polio,TSE não vetou campanha de vacinação contra pol...
301,negacao,Divulgação de pesquisas eleitorais é atribuiçã...,Divulgação de pesquisas eleitorais não é atrib...
400,digitacao,Ciro Gomes peed voto para Bolsonaaro “22” no d...,Ciro Gomes pede voto para Bolsonaro “22” no do...
401,digitacao,Tweet erra ao afimar que seção eleitiral do Pa...,Tweet erra ao afirmar que seção eleitoral do P...


### Conferir o gerador de negação

O gerador é uma regra, e regra erra. Antes de usar a categoria, confira uma amostra de 20 em `results/revisao_negacao.csv`: a `consulta` é o boato que corresponde ao desmentido (`alvo`)? Marque `ok` com `sim` ou `nao`. A fração de `sim` é a **precisão do gerador**, e vai para o README. O arquivo não é sobrescrito depois de revisado.

In [3]:
caminho_revisao = RESULTS / "revisao_negacao.csv"
if not caminho_revisao.exists():
    (teste[teste["categoria"] == "negacao"].sample(20, random_state=48)[["consulta_id", "alvo", "consulta"]]
     .assign(ok="").to_csv(caminho_revisao, index=False))
revisao = pd.read_csv(caminho_revisao, dtype={"ok": str})
revisados = revisao["ok"].str.strip().str.lower().isin(["sim", "nao"])
if revisados.all():
    print(f"Precisão do gerador de negação: {(revisao['ok'].str.strip().str.lower() == 'sim').mean():.0%} (n = {len(revisao)})")
else:
    print(f"⚠️ {(~revisados).sum()} de {len(revisao)} linhas ainda sem revisão em {caminho_revisao}")

⚠️ 20 de 20 linhas ainda sem revisão em results/revisao_negacao.csv


## 2. Normalizar a consulta

Três versões de cada consulta:
- `bruta`: como chegou;
- `enelvo`: Enelvo, com o vocabulário do corpus protegido (ele só mexe em palavras que o catálogo nunca usa);
- `enelvo+apelidos`: primeiro desfaz os apelidos marcados `no_normalizador = sim`, depois o Enelvo.

**Leitura justa dos apelidos:** o mesmo dicionário gera e desfaz os apelidos `sim`, então o resultado deles é um **teto** (circular). Os apelidos `nao` ficam fora do normalizador de propósito e medem o caso real, o de um apelido que o sistema não conhece. O internetês é gerado com o nosso dicionário e corrigido pelo léxico do próprio Enelvo, que é independente.

In [4]:
protegidos = vocabulario_protegido(d["claim"], tabela_internetes)
enelvo = criar_enelvo(protegidos, Path("data/protegidos_enelvo.txt"))
print(len(protegidos), "palavras protegidas")

teste["enelvo"] = [normalizar(t, enelvo) for t in teste["consulta"]]
teste["enelvo+apelidos"] = [normalizar(t, enelvo, apelidos) for t in teste["consulta"]]
# condicional: apelidos sempre; Enelvo só se a consulta tiver cara de internetês
teste["condicional"] = [normalizar_se_preciso(t, enelvo, apelidos, tabela_internetes) for t in teste["consulta"]]
# custo da normalização: consultas que já chegam limpas (a própria alegação)
teste["alvo_enelvo"] = [normalizar(t, enelvo, apelidos) for t in teste["alvo"]]
teste["alvo_condicional"] = [normalizar_se_preciso(t, enelvo, apelidos, tabela_internetes) for t in teste["alvo"]]
teste[["categoria", "consulta", "condicional"]].sample(6, random_state=1)

3897 palavras protegidas


,categoria,consulta,condicional
529,parafrase_real,Vídeo de Bolsonaro cumprimentando apoiadores é...,Vídeo de Bolsonaro cumprimentando apoiadores é...
403,digitacao,Denúncia de policiais soobre suposto problema ...,Denúncia de policiais soobre suposto problema ...
464,digitacao,Erfo mostrado em víeo é de pesquisa contratada...,Erfo mostrado em víeo é de pesquisa contratada...
142,girias,g1 naum publicou q lula disse que 'nem deus ti...,G1 não publicou que Lula disse que ' nem deus ...
521,parafrase_real,Justiça Eleitoral rejeitou candidatura de Lula...,Justiça Eleitoral rejeitou candidatura de Lula...
295,apelido+girias,g1 publicou reportagem em q luladrão diz q edu...,G1 publicou reportagem em que Lula diz que edu...


## 3. Busca e métricas (MiniLM, índice preliminar das 1.852 alegações de 2022)

Acerto = qualquer cópia da alegação-alvo (a mesma checagem publicada por agências diferentes conta como a mesma). A variante `alvo` é a consulta sem reescrita: Recall@1 tem que dar ~100%, e serve como checagem de que a busca está certa. `alvo_enelvo` mostra o quanto a normalização **atrapalha** quem já escreve limpo.

**Na negação, achar o desmentido é o acerto:** o usuário manda "Lula disse X", e o sistema precisa devolver "Lula não disse X". O risco do produto não está na busca, e sim em apresentar o resultado: o veredito tem que ficar claro, e o sistema não pode sugerir que a consulta foi confirmada.

In [5]:
model = SentenceTransformer(MODELO, revision=REVISAO)
relevantes = linhas_com_texto(d["claim"], teste["alvo"])
excluir = linhas_com_texto(d["claim"], teste["excluir_texto"])   # vazio, exceto na paráfrase real

VARIANTES = ["alvo", "alvo_enelvo", "alvo_condicional", "consulta", "enelvo", "enelvo+apelidos", "condicional"]
linhas = []
for variante in VARIANTES:
    Q = model.encode(teste[variante].tolist(), normalize_embeddings=True, batch_size=64)
    teste[f"pos_{variante}"] = posicoes(Q, E, relevantes, excluir)
    grupos = list(teste.groupby("categoria")) + [
        (f"apelido ({'conhecido' if s == 'sim' else 'desconhecido'})", g)
        for s, g in teste[teste["categoria"] == "apelido"].groupby("apelido_no_normalizador")]
    for categoria, g in grupos:
        linhas.append({"variante": "bruta" if variante == "consulta" else variante, "categoria": categoria,
                       **metricas(g[f"pos_{variante}"].to_numpy())})

resultado = pd.DataFrame(linhas)
resultado.to_csv(RESULTS / "normalizacao_consulta.csv", index=False)
resultado.pivot(index="categoria", columns="variante", values="recall@1").round(2)[
    ["alvo", "alvo_condicional", "bruta", "enelvo+apelidos", "condicional"]]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7964.53it/s]

variante,alvo,alvo_condicional,bruta,enelvo+apelidos,condicional
categoria,,,,,
apelido,1.0,1.0,0.98,0.96,0.99
apelido (conhecido),1.0,1.0,0.98,0.97,1.00
apelido (desconhecido),1.0,1.0,0.98,0.95,0.98
apelido+girias,1.0,1.0,0.65,0.99,0.99
digitacao,1.0,1.0,0.94,0.98,0.94
girias,1.0,1.0,0.67,1.00,1.00
negacao,1.0,1.0,0.72,0.65,0.72
parafrase_real,1.0,1.0,0.56,0.56,0.56


In [6]:
# Onde o Enelvo erra: palavras trocadas por outras sem relação
erros = teste[teste["pos_enelvo+apelidos"] > teste["pos_consulta"]]
print(len(erros), "consultas pioraram com a normalização")
erros[["consulta", "enelvo+apelidos", "pos_consulta", "pos_enelvo+apelidos"]].head(10)

24 consultas pioraram com a normalização


,consulta,enelvo+apelidos,pos_consulta,pos_enelvo+apelidos
14,Molusco usou ponto eletrônico em debate da Globo,Lula usou ponto eletrônico em debate da globo,1,2
38,Urna em Mato Grosso com 100% dos votos para Lu...,urna em mato grosso com 100% dos votos para Lu...,1,2
46,Vídeo que mostra Lularápio e Janja na campanha...,vídeo que mostra usurpador e janja na campanha...,1,7
233,"ao dizer no flow q ‘politico tem q mentir’, lu...",ao dizer no flow que ‘ politico tem que mentir...,1,3
301,Divulgação de pesquisas eleitorais é atribuiçã...,divulgação de pesquisas eleitorais é atribuiçã...,1,3
303,William Bonner chamou Lula e Alckmin de ladrões,William bonner chamou Lula e Alckmin de ladrões,2,5
306,Janones disse que Lula vai acabar com Auxílio ...,janones disse que Lula vai acabar com auxílio ...,5,6
307,Randolfe disse que Vitória teve ato pró-Bolson...,randolfe disse que vitória teve ato pró-bolson...,1,3
330,Globo pediu autorização à Justiça para entrevi...,globo pediu autorização à justiça para entrevi...,2,3
350,Vídeo é de comemoração na Maré após vitória de...,vídeo é de comemoração na maré após vitória de...,1,2
